# Merge cell label annotations
We tested two different label transfer methods, one using scANVI (`/00_scanvi-labels`) and one using Seurat (`/03_seurat-labels`). In this notebook, we update our main adata object to include both the scANVI and Seurat labels and add corresponding cell class labels. The Seurat labels are used for the remainder of the analysis, and the scANVI labels are not futher used.

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

In [ ]:
from pathlib import Path
import os
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import warnings
import matplotlib.pyplot as plt

import session_info
import sys

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[1]))

from config.paths import BASE_DIR, METADATA_DIR

# Data directories
input_dir = BASE_DIR / "data/h5ad/export_03/03a_scanvi"
adata_path = input_dir / "adata-scanvi-labels.h5ad"
seurat_labels_path = BASE_DIR / "data/rds/seurat/label_transfer_xenium.csv"

output_dir = BASE_DIR / "data/h5ad/export_03/03b_seurat"

In [ ]:
adata = sc.read_h5ad(adata_path)
seurat_df = pd.read_csv(seurat_labels_path)

# Seurat label transfer annotations

In [ ]:
# Make a copy and set cell barcodes as index
labels = seurat_df.copy()
labels.index = labels["Unnamed: 0"]
labels = labels.drop(columns=["Unnamed: 0"])

# Keep rows present in adata, reindex
shared = labels.index.intersection(adata.obs_names)
labels = labels.loc[shared]
labels = labels.reindex(adata.obs_names)

# Add columns to adata
adata.obs = adata.obs.join(labels)

In [ ]:
adata.obs = adata.obs.rename(columns={"predicted.id": "seurat_labels"})

# Finalize labels

## Neuronal class assignemnt

In [ ]:
#Macrophage_2 indicates a mixed hematopoeitic population from the sequencing dataset; renamed to reflect this.

cols = ["seurat_labels", "scanvi_labels"]

for col in cols:
    adata.obs[col] = adata.obs[col].astype("category")
    adata.obs[col] = adata.obs[col].cat.rename_categories({"Macrophage_2": "Mixed"})
    adata.obs[col] = adata.obs[col].cat.remove_unused_categories()

In [ ]:
label_to_class = {
    # Neurons
    "CGRP-Gamma": "Neuron",
    "CGRP-Eta": "Neuron",
    "CGRP-Zeta": "Neuron",
    "CGRP-Theta": "Neuron",
    "CGRP-Epsilion": "Neuron",
    "CGRP-Alpha": "Neuron",
    "CGRP-Beta": "Neuron",
    "Nonpeptidergic nociceptors": "Neuron",
    "Abeta-RA-LTMR": "Neuron",
    "Abeta-Field": "Neuron",
    "Adelta-LTMR": "Neuron",
    "C-LTMR": "Neuron",
    "TrpM8": "Neuron",
    "Proprioceptors": "Neuron",
    "Sst": "Neuron",

    # Non-neurons
    "Schwann_cell": "Non-neuron",
    "SGC": "Non-neuron",
    "Mixed": "Non-neuron",
    "Vascular_endothelial_1": "Non-neuron",
    "Vascular_endothelial_2": "Non-neuron",
    "Fibroblast_1": "Non-neuron",
    "Fibroblast_2": "Non-neuron",
    "Macrophage_1": "Non-neuron",
    "Pericyte": "Non-neuron",
    "Angiogenic EC": "Non-neuron",
    "Cycling SGC": "Non-neuron",
    "Unassigned": "Non-neuron",
}

In [ ]:
for src, dst in [("scanvi_labels", "scanvi_class"),
                 ("seurat_labels", "seurat_class")]:
    adata.obs[dst] = adata.obs[src].map(label_to_class).astype("category")

In [ ]:
import matplotlib.pyplot as plt

# Count total and neuron fraction
counts = adata.obs["seurat_class"].value_counts()

neuron_count = counts.get("Neuron", 0)
total = counts.sum()
non_neuron_count = total - neuron_count

# Percentages
labels = ["Neuron", "Non-neuron"]
values = [(neuron_count / total) * 100,
          (non_neuron_count / total) * 100]

# --- Minimal plot ---
plt.figure(figsize=(1.5, 2.5), dpi=150)
plt.bar(labels, values, color=["black", "black"], alpha=0.85)   # <-- both gray

# Style: minimalist
plt.ylim(0, 60)
plt.ylabel("% of Dataset", fontsize=10)
plt.xlabel("")
plt.xticks(rotation=45, ha='right', fontsize=9)   # <-- rotated x labels
plt.yticks(fontsize=9)

# Only bottom & left spine
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)

# No tick marks
ax.tick_params(axis='y', length=0)
ax.tick_params(axis='x', length=0)

plt.tight_layout()
plt.show()

# Export

In [ ]:
filename = os.path.join(output_dir, 'adata-merged-labels.h5ad')
os.makedirs(os.path.dirname(filename), exist_ok = True)

adata.write_h5ad(filename, compression='gzip')